In [1]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
sys.path.append("/content/drive/MyDrive/tcc")
from src import config

import pandas as pd
from sklearn.model_selection import train_test_split

pares = pd.read_csv(config.PARES_CSV)
df = pd.read_csv(config.DATASET_CSV)
print(len(pares), "pares |", len(df), "linhas no dataset")

Mounted at /content/drive
1000 pares | 5000 linhas no dataset


In [2]:
if "split" in pares.columns:
    raise RuntimeError("A divisão já foi feita. Não rode esta célula de novo.")

treino, resto = train_test_split(
    pares, test_size=0.30, stratify=pares["gerador"], random_state=config.SEED)
val, teste = train_test_split(
    resto, test_size=0.50, stratify=resto["gerador"], random_state=config.SEED)

split_do_id = {
    **dict.fromkeys(treino["id"], "treino"),
    **dict.fromkeys(val["id"], "val"),
    **dict.fromkeys(teste["id"], "teste"),
}
pares["split"] = pares["id"].map(split_do_id)
df["split"] = df["id"].map(split_do_id)

print(pares["split"].value_counts())

split
treino    700
teste     150
val       150
Name: count, dtype: int64


In [3]:
print("Gerador por conjunto:")
print(pd.crosstab(pares["gerador"], pares["split"]))

print("\nCâmera por conjunto:")
print(pd.crosstab(pares["camera"], pares["split"]))

print("\nLinhas do dataset por conjunto e papel:")
print(df.groupby(["split", "papel", "versao", "classe"]).size())

ids_por_conjunto = df.groupby("id")["split"].nunique()
print("\nIDs em mais de um conjunto:", (ids_por_conjunto > 1).sum())

Gerador por conjunto:
split                 teste  treino  val
gerador                                 
dalle2                   21     100   22
dalle3                   21     100   22
midjourney-v5            22     100   21
stable-diffusion-1-3     22     100   21
stable-diffusion-1-4     22     100   21
stable-diffusion-2       21     100   22
stable-diffusion-xl      21     100   21

Câmera por conjunto:
split        teste  treino  val
camera                         
Nikon D40        0       7    1
Nikon D7000    119     497  113
Nikon D90       31     196   36

Linhas do dataset por conjunto e papel:
split   papel      versao    classe
teste   ood        original  ia        150
        principal  jpeg90    ia        150
                             real      150
                   original  ia        150
                             real      150
treino  ood        original  ia        700
        principal  jpeg90    ia        700
                             real      700
      

In [4]:
pares.to_csv(config.PARES_CSV, index=False)
df.to_csv(config.DATASET_CSV, index=False)
print("Salvo.")
print(pd.read_csv(config.DATASET_CSV)["split"].value_counts())

Salvo.
split
treino    3500
teste      750
val        750
Name: count, dtype: int64
